# R2-Dreamer Baseline Comparison: Random vs Trained

**Goal:** Side-by-side comparison of random agent vs trained R2-Dreamer on the 4 success criteria:
1. World model learns (loss curves — trained only)
2. Agent explores (action distribution doesn't collapse)
3. Reward improves (reward distribution comparison)
4. Qualitative (top-down trajectory maps)

| Input | Source |
|-------|--------|
| Trained eval | `eval_habitat.py --checkpoint <ckpt> --episodes 50 --output <path>` |
| Random eval  | `eval_habitat.py --random --episodes 50 --output <path>` |
| Training CSV | `output/r2dreamer-habitat-baseline/*/metrics.csv` |

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import LineCollection

plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (12, 5)})

In [ ]:
# --- Configure paths ---
RUN_DIR = Path("../../../output/r2dreamer-habitat-baseline/sanity-50k")
TRAINED_EVAL = RUN_DIR / "eval_trained.json"
RANDOM_EVAL = RUN_DIR / "eval_random.json"
METRICS_CSV = RUN_DIR / "metrics.csv"

assert TRAINED_EVAL.exists(), f"Not found: {TRAINED_EVAL}"
assert RANDOM_EVAL.exists(), f"Not found: {RANDOM_EVAL}"

with open(TRAINED_EVAL) as f:
    trained = json.load(f)["results"]
with open(RANDOM_EVAL) as f:
    random_results = json.load(f)["results"]

print(f"Trained: {len(trained)} episodes")
print(f"Random:  {len(random_results)} episodes")

if METRICS_CSV.exists():
    df = pd.read_csv(METRICS_CSV)
    print(f"Training CSV: {len(df)} rows")
else:
    df = None
    print("No training CSV found — loss curves will be skipped")

## Criterion 1: World Model Learns (Loss Curves)

In [ ]:
def extract_metric(df, name):
    sub = df[df["metric"] == name]
    return sub["step"].values, sub["value"].values.astype(float)

if df is not None:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, metric, label in zip(axes, ["loss/dyn", "loss/rep", "total_loss"],
                                  ["Dynamics Loss", "Representation Loss", "Total Loss"]):
        steps, vals = extract_metric(df, metric)
        ax.plot(steps, vals, alpha=0.3, linewidth=0.5, color="gray")
        if len(vals) >= 20:
            rolling = pd.Series(vals).rolling(20).mean().values
            ax.plot(steps, rolling, linewidth=2, color="blue", label="rolling-20")
        ax.set_xlabel("Step")
        ax.set_title(label)
        ax.legend()
        ax.grid(True, alpha=0.3)
    fig.suptitle("Criterion 1: World Model Losses", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped — no training CSV")

## Criterion 2: Agent Explores (Action Distribution)

In [ ]:
ACTION_NAMES = ["STOP", "MOVE_FORWARD", "TURN_LEFT", "TURN_RIGHT"]

def get_action_pcts(results):
    total = {n: 0 for n in ACTION_NAMES}
    for r in results:
        for name, count in r["action_counts"].items():
            total[name] += count
    s = sum(total.values())
    return {n: total[n] / s * 100 for n in ACTION_NAMES}

trained_pcts = get_action_pcts(trained)
random_pcts = get_action_pcts(random_results)

x = np.arange(len(ACTION_NAMES))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
bars1 = ax.bar(x - width/2, [trained_pcts[n] for n in ACTION_NAMES], width,
               label="Trained", color="steelblue")
bars2 = ax.bar(x + width/2, [random_pcts[n] for n in ACTION_NAMES], width,
               label="Random", color="lightcoral")

for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"{bar.get_height():.1f}%", ha="center", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(ACTION_NAMES)
ax.set_ylabel("Percentage")
ax.set_title("Criterion 2: Action Distribution — Trained vs Random")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
ax.axhline(y=25, color="gray", linestyle=":", alpha=0.5, label="uniform")
plt.tight_layout()
plt.show()

# Entropy comparison
def action_entropy(pcts):
    p = np.array([pcts[n] / 100 for n in ACTION_NAMES])
    p = p[p > 0]
    return -np.sum(p * np.log2(p))

print(f"Action entropy — Trained: {action_entropy(trained_pcts):.2f} bits, "
      f"Random: {action_entropy(random_pcts):.2f} bits (max=2.0)")

## Criterion 3: Reward Improves

In [ ]:
trained_rewards = [r["reward"] for r in trained]
random_rewards = [r["reward"] for r in random_results]
trained_steps = [r["steps"] for r in trained]
random_steps = [r["steps"] for r in random_results]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Reward histogram
bins = np.linspace(min(min(trained_rewards), min(random_rewards)),
                   max(max(trained_rewards), max(random_rewards)), 25)
axes[0].hist(trained_rewards, bins=bins, alpha=0.6, label="Trained", color="steelblue")
axes[0].hist(random_rewards, bins=bins, alpha=0.6, label="Random", color="lightcoral")
axes[0].axvline(np.mean(trained_rewards), color="blue", linestyle="--", linewidth=2)
axes[0].axvline(np.mean(random_rewards), color="red", linestyle="--", linewidth=2)
axes[0].set_xlabel("Episode Reward")
axes[0].set_ylabel("Count")
axes[0].set_title("Reward Distribution")
axes[0].legend()

# Episode length histogram
axes[1].hist(trained_steps, bins=20, alpha=0.6, label="Trained", color="steelblue")
axes[1].hist(random_steps, bins=20, alpha=0.6, label="Random", color="lightcoral")
axes[1].set_xlabel("Episode Steps")
axes[1].set_ylabel("Count")
axes[1].set_title("Episode Length Distribution")
axes[1].legend()

# Box plot comparison
bp = axes[2].boxplot([trained_rewards, random_rewards], labels=["Trained", "Random"],
                     patch_artist=True)
bp["boxes"][0].set_facecolor("steelblue")
bp["boxes"][1].set_facecolor("lightcoral")
axes[2].set_ylabel("Episode Reward")
axes[2].set_title("Reward Comparison")
axes[2].grid(True, alpha=0.3, axis="y")

fig.suptitle("Criterion 3: Reward — Trained vs Random", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Summary stats
print(f"{'Metric':<20s} {'Trained':>10s} {'Random':>10s} {'Delta':>10s}")
print("-" * 52)
for name, t_vals, r_vals in [
    ("Mean reward", trained_rewards, random_rewards),
    ("Mean steps", trained_steps, random_steps),
]:
    tm, rm = np.mean(t_vals), np.mean(r_vals)
    print(f"{name:<20s} {tm:>10.2f} {rm:>10.2f} {tm - rm:>+10.2f}")

t_sr = np.mean([r["success"] for r in trained]) * 100
r_sr = np.mean([r["success"] for r in random_results]) * 100
print(f"{'Success rate (%)':<20s} {t_sr:>10.1f} {r_sr:>10.1f} {t_sr - r_sr:>+10.1f}")

## Criterion 4: Qualitative — Trajectory Maps (Trained vs Random)

Side-by-side top-down maps for the first N episodes. Requires trajectory data in eval JSONs.

In [ ]:
has_traj = ("trajectory" in trained[0]) and ("trajectory" in random_results[0])

if has_traj:
    import habitat_sim

    def create_topdown_map(scene_path, resolution=0.05):
        backend_cfg = habitat_sim.SimulatorConfiguration()
        backend_cfg.scene_id = scene_path
        backend_cfg.load_semantic_mesh = True
        agent_cfg = habitat_sim.agent.AgentConfiguration()
        cfg = habitat_sim.Configuration(backend_cfg, [agent_cfg])
        sim = habitat_sim.Simulator(cfg)

        agent_y = sim.get_agent(0).get_state().position[1]
        pf = sim.pathfinder
        bounds = pf.get_bounds()
        x_min, z_min = bounds[0][0], bounds[0][2]
        x_max, z_max = bounds[1][0], bounds[1][2]
        xs = np.arange(x_min, x_max, resolution)
        zs = np.arange(z_min, z_max, resolution)
        H, W = len(zs), len(xs)

        image = np.ones((H, W, 3)) * 0.95
        for zi, z in enumerate(zs):
            for xi, x in enumerate(xs):
                if pf.is_navigable(np.array([x, agent_y, z]), max_y_delta=0.5):
                    image[zi, xi] = [0.85, 0.85, 0.85]

        cmap = plt.cm.tab20
        seen = {}
        for region in sim.semantic_scene.regions:
            cat = region.category.name() if region.category else "unknown"
            if cat not in seen:
                seen[cat] = cmap(len(seen) % 20)[:3]
            color = seen[cat]
            aabb = region.aabb
            rx0, rx1 = aabb.center[0] - aabb.sizes[0]/2, aabb.center[0] + aabb.sizes[0]/2
            rz0, rz1 = aabb.center[2] - aabb.sizes[2]/2, aabb.center[2] + aabb.sizes[2]/2
            xi0 = max(0, int((rx0 - x_min) / resolution))
            xi1 = min(W, int((rx1 - x_min) / resolution))
            zi0 = max(0, int((rz0 - z_min) / resolution))
            zi1 = min(H, int((rz1 - z_min) / resolution))
            for z_i in range(zi0, zi1):
                for x_i in range(xi0, xi1):
                    if np.allclose(image[z_i, x_i], [0.85, 0.85, 0.85]):
                        image[z_i, x_i] = color

        sim.close()
        return {"image": image, "x_min": x_min, "x_max": x_max,
                "z_min": z_min, "z_max": z_max, "resolution": resolution}

    def plot_trajectory(ax, r, map_data, title):
        ax.imshow(map_data["image"], origin="lower",
                  extent=[map_data["x_min"], map_data["x_max"],
                          map_data["z_min"], map_data["z_max"]])
        traj = np.array(r["trajectory"])
        points = np.column_stack([traj[:, 0], traj[:, 2]]).reshape(-1, 1, 2)
        segments = np.concatenate([points[:-1], points[1:]], axis=1)
        colors = plt.cm.coolwarm(np.linspace(0, 1, len(segments)))
        ax.add_collection(LineCollection(segments, colors=colors, linewidths=1.5, alpha=0.8))
        start = r["start_position"]
        stop = r["trajectory"][-1]
        ax.plot(start[0], start[2], "^", color="lime", ms=10, mec="black", mew=0.5, zorder=5)
        ax.plot(stop[0], stop[2], "s", color="red", ms=8, mec="black", mew=0.5, zorder=5)
        for gp in r["goal_positions"]:
            ax.plot(gp[0], gp[2], "*", color="gold", ms=14, mec="black", mew=0.5, zorder=5)
        ax.set_title(title, fontsize=9)
        ax.set_aspect("equal")

    # Plot first N episodes side-by-side (trained left, random right)
    N = min(6, len(trained), len(random_results))
    scene_maps = {}

    fig, axes = plt.subplots(N, 2, figsize=(10, 5 * N))
    if N == 1:
        axes = axes.reshape(1, -1)

    for i in range(N):
        t, r = trained[i], random_results[i]
        scene_id = t["scene_id"]
        if scene_id not in scene_maps:
            scene_maps[scene_id] = create_topdown_map(scene_id)
        md = scene_maps[scene_id]

        cat = t.get("object_category", "?")
        plot_trajectory(axes[i, 0], t, md,
                        f"Trained Ep{t['episode']} [{cat}] r={t['reward']:.1f}")
        # Random may be on different scene — load if needed
        r_scene = r["scene_id"]
        if r_scene not in scene_maps:
            scene_maps[r_scene] = create_topdown_map(r_scene)
        plot_trajectory(axes[i, 1], r, scene_maps[r_scene],
                        f"Random Ep{r['episode']} [{r.get('object_category','?')}] r={r['reward']:.1f}")

    axes[0, 0].set_title("TRAINED\n" + axes[0, 0].get_title(), fontsize=10, fontweight="bold")
    axes[0, 1].set_title("RANDOM\n" + axes[0, 1].get_title(), fontsize=10, fontweight="bold")
    fig.suptitle("Criterion 4: Trajectory Comparison", fontsize=14, fontweight="bold")
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()
else:
    print("No trajectory data — re-run eval_habitat.py with updated version")

## Verdict

Check each criterion:
- [ ] **World model learns**: dyn/rep losses decrease and stabilize
- [ ] **Agent explores**: action distribution differs from random, no single-action collapse
- [ ] **Reward improves**: trained agent mean reward > random agent mean reward
- [ ] **Qualitative**: trained trajectories show purposeful movement toward goals